# Prompt engineering exercise: topic extraction

Start from the baseline prompt `What topics are in here?` and iterate on it.
Add new variants to the `variants` dict and re-run the eval cell to compare scores.


In [1]:
import contextlib
import io
import textwrap
from statistics import mean

from evals import PromptEvaluator

from chat import add_user_message, chat

evaluator = PromptEvaluator(max_concurrent_tasks=16)

In [2]:
dataset = evaluator.generate_dataset(
    task_description="Extract topics from a passage as a list of strings",
    prompt_inputs_spec={
        "content": "One paragraph of English text containing several topics",
    },
    output_file="dataset.json",
    num_cases=4,
)

Generated 1/4 test cases
Generated 2/4 test cases
Generated 3/4 test cases
Generated 4/4 test cases


In [3]:
def block(text):
    return textwrap.dedent(text).strip("\n")


content_block = block("""
    <content>
    {content}
    </content>
""")

qualities_block = block("""
    The output must have the following qualities:
    - It should be formatted as a JSON array of strings, without any additional text before or after.
    - Each entry of the JSON array should refer to a SINGLE topic mentioned in the article.
    - Two different entries shouldn't refer to the same topic.
    Regarding granularity, consider the following hints:
    - Minimalistic texts may only have one or two topics.
    - Ambiguous or implicit topics may require inferring a broader topic.
    - Specific items can sometimes be elevated to a topic-level status.
""")

sample_input_block = block("""
    <sample_input>
    content: Climate change represents one of the most pressing challenges of our time, driven by greenhouse gas
    emissions from fossil fuel combustion, industrial agriculture, and deforestation. Rising global temperatures trigger
    cascading effects: sea levels are rising due to thermal expansion and melting ice sheets, threatening coastal
    communities and island nations. Extreme weather patterns including hurricanes, droughts, and floods are becoming more
    frequent and severe, impacting food security and agricultural productivity worldwide. Ocean acidification from
    increased CO2 absorption is decimating marine ecosystems and coral reefs. Meanwhile, renewable energy technologies
    like solar and wind power are advancing rapidly, offering sustainable alternatives to coal and natural gas. Carbon
    capture and storage methods, reforestation initiatives, and policy frameworks such as carbon pricing are emerging as
    critical mitigation strategies. Public health implications include heat-related illnesses, disease vector
    expansion, and mental health impacts from climate anxiety. International cooperation through agreements like the
    Paris Accord aims to coordinate global emissions reduction targets and climate adaptation funding.
    </sample_input>
""")

ideal_output_block = block("""
    <ideal_output>
    ```json
    [
      "Greenhouse gas emissions",
      "Fossil fuel combustion",
      "Industrial agriculture",
      "Deforestation",
      "Rising global temperatures",
      "Sea level rise",
      "Thermal expansion",
      "Melting ice sheets",
      "Coastal communities and island nations",
      "Extreme weather patterns",
      "Hurricanes",
      "Droughts",
      "Floods",
      "Food security",
      "Agricultural productivity",
      "Ocean acidification",
      "Marine ecosystems",
      "Coral reefs",
      "Solar power",
      "Wind power",
      "Renewable energy technologies",
      "Coal and natural gas",
      "Carbon capture and storage",
      "Reforestation",
      "Carbon pricing",
      "Heat-related illnesses",
      "Disease vector expansion",
      "Climate anxiety",
      "Mental health impacts",
      "Paris Accord",
      "Global emissions reduction targets",
      "Climate adaptation funding",
      "International cooperation"
    ]
    ```
    </ideal_output>
    The solution meets all mandatory requirements: it's a JSON array of strings with topics only and no extra commentary.
    For secondary criteria, it far exceeds the '8+ distinct topics' requirement with 33 topics, maintains accuracy by
    correctly identifying concepts from the passage, and demonstrates completeness by covering causes, effects,
    solutions, and policy aspects. While the granularity could be debated (some topics are quite specific while others
    are broader), the task doesn't specify a preferred level of abstraction. The solution is thorough and technically
    correct.
""")

example_block = "\n".join(
    [
        "Here is a sample input paired with an ideal output:",
        "<example>",
        sample_input_block,
        ideal_output_block,
        "</example>",
    ]
)

# Prompt variants. Each template uses the keys declared in `prompt_inputs_spec`.
# Add new variants here as you iterate.

# 1. Simple
task_statement_1 = "What topics are in here?"
prompt_1_simple = task_statement_1 + "\n\n" + content_block

# 2. Clear and direct
task_statement_2 = "Generate a JSON array of strings with the topics from the article below."
prompt_2_clear_and_direct = task_statement_2 + "\n\n" + content_block

# 3. Be specific by listing qualities the output must have.
prompt_3_list_qualities = prompt_2_clear_and_direct + "\n\n" + qualities_block

# 4. Add examples.
prompt_4_add_examples = prompt_3_list_qualities + "\n\n" + example_block

variants = {
    "1_simple": prompt_1_simple,
    "2_clear_and_direct": prompt_2_clear_and_direct,
    "3_list_qualities": prompt_3_list_qualities,
    "4_add_examples": prompt_4_add_examples,
}

In [4]:
# Factory: takes a prompt template, returns a `run_prompt(prompt_inputs)` function
# that `evaluator.run_evaluation` can call for each test case.
def make_run_prompt(template):
    def run_prompt(prompt_inputs):
        messages = []
        add_user_message(messages, template.format(**prompt_inputs))
        return chat(messages)

    return run_prompt

In [5]:
extra_criteria = block("""
    - Contains a JSON array of strings, containing each topic mentioned in the article.
    - The strings should contain only a topic without any extra commentary.
    - Response should contain the JSON array and nothing else.
""")

scores = {}
for name, template in variants.items():
    with contextlib.redirect_stdout(io.StringIO()):
        results = evaluator.run_evaluation(
            run_prompt_function=make_run_prompt(template),
            dataset_file="dataset.json",
            extra_criteria=extra_criteria,
            json_output_file=f"output_{name}.json",
            html_output_file=f"output_{name}.html",
        )
    scores[name] = mean(r["score"] for r in results)

for name, score in scores.items():
    print(f"  {name}: {score:.2f}")

  1_simple: 1.00
  2_clear_and_direct: 7.50
  3_list_qualities: 7.75
  4_add_examples: 7.75
